# Notebook 2 — Create the Labels

## Objective

The goal of this notebook is to create the target variable for the delivery prediction task.

An order is labeled as **late** when its actual delivery date is after the estimated delivery date. Otherwise, the order is labeled as **on_time**.

This notebook also validates the generated labels on real orders and examines the class distribution.

### Output

The final output of this notebook is a labeled dataset with one row per order, saved as:

`../Artifacts/labeled_table.csv`

In [1]:
import pandas as pd
import numpy as np 

## Load the ML Table

The input dataset for this notebook is the ML table created in Notebook 1. Each row represents one order and contains the information needed to create the delivery label.

In [2]:
ml_table = pd.read_csv("../Artifacts/ml_table.csv")

ml_table.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,number_of_items,total_price,total_freight,number_of_payments,total_payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,29.99,8.72,3.0,38.71
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,118.70,22.76,1.0,141.46
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1.0,159.90,19.22,1.0,179.12
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,45.00,27.20,1.0,72.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,1.0,19.90,8.72,1.0,28.62


## Prepare Delivery Date Columns

The delivery label is based on comparing the actual delivery date with the estimated delivery date. Therefore, both columns must be converted to `datetime` before creating the label.

In [3]:
date_columns = [
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

ml_table[date_columns] = ml_table[date_columns].apply(
    pd.to_datetime
)

In [4]:
ml_table[date_columns].dtypes

order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## Handle Orders Without an Actual Delivery Date

Orders without an actual delivery date cannot be reliably labeled as `late` or `on_time`. Therefore, these orders will be excluded from the labeled dataset.

The original `ml_table` remains unchanged, and a separate `labeled_table` is created using only orders with a non-missing actual delivery date.

In [5]:
missing_actual_delivery = (
    ml_table["order_delivered_customer_date"]
    .isna()
    .sum()
)

print(
    f"Orders without an actual delivery date: "
    f"{missing_actual_delivery:,}"
)

Orders without an actual delivery date: 2,965


In [6]:
labeled_table = ml_table[
    ml_table["order_delivered_customer_date"].notna()
].copy()

print(f"Labeled orders: {len(labeled_table):,}")

Labeled orders: 96,476


## Create the Target Label

The target variable is created by comparing the actual delivery date with the estimated delivery date.

Because the estimated delivery column does not contain a specific delivery time, the comparison is performed at the date level.

* `late`: actual delivery date is after the estimated delivery date.
* `on_time`: actual delivery date is on or before the estimated delivery date.

In [7]:
actual_date = labeled_table[
    "order_delivered_customer_date"
].dt.normalize()

estimated_date = labeled_table[
    "order_estimated_delivery_date"
].dt.normalize()

labeled_table["delivery_label"] = np.where(
    actual_date > estimated_date,
    "late",
    "on_time"
)

## Validate the Labels

To verify that the target variable was created correctly, a sample of orders from both classes is reviewed.

The validation checks that:

* `late` orders have an actual delivery date after the estimated delivery date.
* `on_time` orders have an actual delivery date on or before the estimated delivery date.
* Orders delivered on the same date as the estimated delivery date are correctly labeled as `on_time`.

In [9]:
label_sample = (
    labeled_table
    .groupby("delivery_label")
    .sample(n=5, random_state=42)
)

label_sample[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_label"
    ]
]

,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_label
82938,c1eb421afe27745d1845f27fdbd14e45,2018-04-10 00:41:36,2018-04-03,late
43761,114c9407eb2b873d07c1b3bf91e35a9c,2017-10-16 18:38:57,2017-10-06,late
43701,4ef685e6dcade551ce14cb3783ca664d,2018-02-23 01:17:38,2018-02-16,late
3441,b357dd0e3d6572e28d060309c0ab339f,2017-12-20 15:44:48,2017-12-11,late
57173,ccbabfa81bc2a6cb8f2326278a01d55c,2018-03-26 23:33:53,2018-03-16,late
6985,189ce876121fdc47b810933fb0b4f4bd,2017-12-04 20:09:09,2017-12-15,on_time
29912,020c01e41104519842f8a15605af646c,2017-02-14 14:47:33,2017-03-09,on_time
22737,30795f13abba18bf1c6866eb7fa63d52,2017-12-21 13:08:36,2018-01-12,on_time
17218,bad2c5064380acc11819dde0237b47a7,2018-06-22 19:18:43,2018-07-03,on_time
22190,71408894cb2412e1f48530b6ec0a2e4b,2018-07-25 21:16:34,2018-08-01,on_time


In [10]:
same_day_orders = labeled_table[
    labeled_table["order_delivered_customer_date"].dt.normalize()
    == labeled_table["order_estimated_delivery_date"].dt.normalize()
]

same_day_orders[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_label"
    ]
].head(10)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,delivery_label
35,8563039e855156e48fccee4d611a3196,2018-03-20 00:59:25,2018-03-20,on_time
152,1d067305b599c1e0dceb3864056ea527,2018-03-09 21:52:36,2018-03-09,on_time
226,c82a457d646f77c8f151e12a3e517ed2,2018-08-15 15:02:09,2018-08-15,on_time
269,2c3a5e5f5bc3ae78ed5748461e62d47d,2018-02-06 15:49:01,2018-02-06,on_time
357,6cc4eb51469dec50a6130fe1cd07b7f9,2017-05-19 15:55:47,2017-05-19,on_time
385,5342acb7b183acec77f59e4d78f7c804,2018-04-24 16:10:44,2018-04-24,on_time
404,884b1394fc8888e6a877df86eb19e74c,2018-03-29 17:32:05,2018-03-29,on_time
655,2f4545bda1b1c794a9a35a3fbe4257ba,2018-06-05 15:38:50,2018-06-05,on_time
753,1a4cc55b483d875e3b8bbd64a1b31bba,2018-05-09 15:08:44,2018-05-09,on_time
762,ed24759a0ae96582e88fdca0c0997d22,2018-04-23 14:57:48,2018-04-23,on_time


## Class Distribution

The class distribution is examined to understand the balance between late and on-time orders.

This is important because an imbalanced target can affect model training and evaluation. In particular, accuracy alone may be misleading when one class is much more common than the other.

In [11]:
class_distribution = (
    labeled_table["delivery_label"]
    .value_counts()
    .rename_axis("delivery_label")
    .reset_index(name="count")
)

class_distribution["percentage"] = (
    class_distribution["count"]
    / class_distribution["count"].sum()
    * 100
)

class_distribution

,delivery_label,count,percentage
0,on_time,89941,93.226295
1,late,6535,6.773705


### Findings

The dataset is clearly imbalanced:

* `on_time`: 89,941 orders (93.23%)
* `late`: 6,535 orders (6.77%)

The `late` class is the minority class. Therefore, accuracy alone will not be sufficient for evaluating the model, as a model could achieve high accuracy by predicting most orders as `on_time`.

This class imbalance will be considered during model training and evaluation.

## Final Validation

Before saving the labeled dataset, final validation checks are performed to ensure that:

* Each row represents one order.
* `order_id` values are unique.
* Every labeled order has a valid target value.
* The final dataset contains the expected number of orders.

In [12]:
print("Number of rows:", len(labeled_table))
print("Unique orders:", labeled_table["order_id"].nunique())
print("Missing labels:", labeled_table["delivery_label"].isna().sum())

Number of rows: 96476
Unique orders: 96476
Missing labels: 0


## Save the Labeled Dataset

The validated labeled dataset is saved as an artifact for use in the next stages of the project.

The output contains only orders with a known actual delivery date and includes the `delivery_label` target variable.

In [13]:
labeled_table.to_csv(
    "../Artifacts/labeled_table.csv",
    index=False
)